In [13]:
%reload_ext autoreload
%autoreload 2

import getpass
import os
import sys
from pathlib import Path

import mlflow
import openai
import pandas as pd
from mistralai.client import MistralClient
from mistralai.models.chat_completion import ChatMessage


In [14]:
os.environ["MLFLOW_TRACKING_URI"] = "http://127.0.0.1:5000"

# os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter OPENAI API key (for evaluation):")
os.environ["MISTRAL_API_KEY"] = getpass.getpass("Enter MISTRAL API key (for model):")

mlflow.set_experiment("LevelUp-Question-Generation-Evaluation")

<Experiment: artifact_location='mlflow-artifacts:/176946843049457413', creation_time=1750265873360, experiment_id='176946843049457413', last_update_time=1750265873360, lifecycle_stage='active', name='LevelUp-Question-Generation-Evaluation', tags={}>

In [15]:
eval_df = pd.DataFrame(
    {
        "inputs": [
            "Generate an easy difficulty problem about arrays.",
            "Create a medium difficulty problem related to hashmaps.",
            "Generate a hard difficulty problem about dynamic programming.",
            "Create a medium difficulty problem focusing on graph traversal.",
            "Generate an easy problem about string manipulation.",
        ],
        "ground_truth": [
            "Given a sorted array of distinct integers and a target value, return the index if the target is found. If not, return the index where it would be if it were inserted in order.",
            "Given an array of integers, find the first recurring character. If no character repeats, return null.",
            "Given a rod of length n inches and an array of prices that contains prices of all pieces of size smaller than n, determine the maximum value obtainable by cutting up the rod and selling the pieces.",
            "Given a binary tree, find the lowest common ancestor (LCA) of two given nodes in the tree.",
            "Given a string, reverse only the vowels of the string and return it.",
        ],
    }
)

In [16]:
from mlflow.metrics.genai import EvaluationExample, make_genai_metric

helpfulness_metric = make_genai_metric(
    name="helpfulness",
    definition=(
        "Helpfulness assesses how well the generated interview problem meets the user's request. "
        "A helpful problem is clear, relevant to the specified topic and difficulty, and serves as a good "
        "basis for a technical interview."
    ),
    grading_prompt=(
        "Helpfulness Score: Evaluate the generated interview problem based on the following criteria:\n"
        "- Score 1: The problem is completely irrelevant to the topic, unclear, or technically incorrect.\n"
        "- Score 2: The problem is related to the topic but is poorly phrased, the difficulty is mismatched, or it is a very common, unoriginal question.\n"
        "- Score 3: The problem is relevant and the difficulty is appropriate, but it could be clearer or more interesting.\n"
        "- Score 4: The problem is clear, relevant, at the correct difficulty, and is a solid interview question.\n"
        "- Score 5: The problem is exceptionally clear, creative, perfectly matches the topic and difficulty, and would be an excellent and insightful interview question."
    ),
    examples=[
        EvaluationExample(
            input="Generate a medium difficulty problem about sorting.",
            output=(
                "Write a function to sort a list of numbers."
            ),
            score=2,
            justification=(
                "The problem is on-topic but is too simple for a medium difficulty and is a very basic question."
            ),
        )
    ],
    version="v1",
    model="mistral:/mistral-tiny",
    parameters={"temperature": 0.0},
    grading_context_columns=[],
    aggregations=["mean", "variance", "p90"],
    greater_is_better=True,
)

print(helpfulness_metric)

EvaluationMetric(name=helpfulness, greater_is_better=True, long_name=helpfulness, version=v1, metric_details=
Task:
You must return the following fields in your response in two lines, one below the other:
score: Your numerical score for the model's helpfulness based on the rubric
justification: Your reasoning about the model's helpfulness score

You are an impartial judge. You will be given an input that was sent to a machine
learning model, and you will be given an output that the model produced. You
may also be given additional information that was used by the model to generate the output.

Your task is to determine a numerical score called helpfulness based on the input and output.
A definition of helpfulness and a grading rubric are provided below.
You must use the grading rubric to determine your score. You must also justify your score.

Examples could be included below for reference. Make sure to use them as references and to
understand them before completing the task.

Input:
{i

In [17]:

with mlflow.start_run() as run:
    system_prompt = "You are an assistant that creates technical interview problems. The user will specify a topic and a difficulty level. Generate a concise and clear problem statement."

    levelup_qa_model = mlflow.pyfunc.log_model(
        python_model="../src/mistral_wrapper.py",
        artifact_path="model",
        input_example=eval_df[["inputs"]].head(1)
    )

    results = mlflow.evaluate(
        model=levelup_qa_model.model_uri,
        data=eval_df,
        targets="ground_truth",
        model_type="question-answering",
        feature_names=["inputs"],
        evaluators="default",
        extra_metrics=[helpfulness_metric],
    )

print("Evaluation Metrics:")
print(results.metrics)

print("\nEvaluation Results Table:")


2025/06/19 00:20:29 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
C:\Users\paypa\OneDrive\Desktop\Project\MlOps\experiments-for-modern-ai-and-mlops\.venv\Lib\site-packages\mlflow\pyfunc\utils\data_validation.py:186: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(
2025/06/19 00:20:29 INFO mlflow.pyfunc: Inferring model signature from input example
2025/06/19 00:20:32 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
C:\Users\paypa\OneDrive\Desktop\Project\MlOps\experiments-for-modern-ai-and-mlops\.venv\Lib\site-packages\mlflow\pyfunc\utils\data_validation.py:186: UserWarning: Add type hints to the `predict` method to enable data val

🏃 View run tasteful-pug-835 at: http://127.0.0.1:5000/#/experiments/176946843049457413/runs/f0a4860da5584ae2adfebc360250c9f2
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/176946843049457413
Evaluation Metrics:
{'exact_match/v1': 0.0, 'helpfulness/v1/mean': np.float64(4.0), 'helpfulness/v1/variance': np.float64(0.0), 'helpfulness/v1/p90': np.float64(4.0)}

Evaluation Results Table:


In [18]:
results.tables["eval_results_table"]

,inputs,ground_truth,outputs,token_count,helpfulness/v1/score,helpfulness/v1/justification
0,Generate an easy difficulty problem about arrays.,Given a sorted array of distinct integers and ...,"Problem: Given an array of numbers, find the s...",72,4,The problem is relevant to the topic of arrays...
1,Create a medium difficulty problem related to ...,"Given an array of integers, find the first rec...","Problem: Given an array of integers, find the ...",124,4,The problem is relevant to the topic of hashma...
2,Generate a hard difficulty problem about dynam...,Given a rod of length n inches and an array of...,Problem: Given an array of integers representi...,143,4,"The problem is clear, relevant, at the correct..."
3,Create a medium difficulty problem focusing on...,"Given a binary tree, find the lowest common an...",Problem: Given a directed graph with no self-l...,105,4,The problem is relevant to the topic of graph ...
4,Generate an easy problem about string manipula...,"Given a string, reverse only the vowels of the...",Write a function that accepts a string as an a...,53,4,The problem is relevant to the topic of string...
